<a href="https://colab.research.google.com/github/MichalSlowakiewicz/Visual-Recognition/blob/master/LAB_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Contrastive Learning with SimCLR
Based on [this](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial17/SimCLR.html) tutorial by Phillip Lippe (see also [JAX](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/JAX/tutorial17/SimCLR.html) version) from [the University of Amsterdam deep learning course](https://uvadlc.github.io/).<br>
It is itself based on [SimCLR](https://simclr.github.io/) (Ting Chen et al., ICML'2020, whence the figure; follow-up: [SimCLRv2](https://arxiv.org/abs/2006.10029)).

<center width="100%"><img src="https://github.com/phlippe/uvadlc_notebooks/blob/master/docs/tutorial_notebooks/tutorial17/simclr_contrastive_learning.png?raw=1" width="700px"></center>

***SimCLR*** is a simple and influential method for contrastive learning.

***Contrastive learning*** is an ***unsupervised*** (a.k.a ***self-supervised***) approach to learning representations from data: in this setting we have no labels, but the images themselves still contains a lot of information from which we can learn.
In this notebook our goal will be to pretrain a model from such unlabeled data, so that it can be quickly fine-tuned to any image recognition task afterwards.

Specifically, we train a model to output vectors that are:
* distant from each other – for distinct images, but
* close together – for augmented versions of the same image, like disjoint crops.

This way we push the model to learn non-trivial representations that are invariant to chosen augmentations, and hopefully capture the higher-level, semantic content of the image.


## Imports

In [ ]:
%pip install lightning==2.6.1 --quiet

In [ ]:
import logging
import os
import warnings
from copy import deepcopy
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.datasets
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint
from torch import Tensor
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision.transforms import v2
from tqdm import tqdm

%load_ext tensorboard

In [ ]:
NUM_WORKERS = min(os.cpu_count() or 2, 8)
print("Number of workers:", NUM_WORKERS)

DEVICE = torch.accelerator.current_accelerator(True) or torch.device("cpu")
torch.set_default_device(DEVICE)
print(DEVICE)

In [ ]:
class TipFilter(logging.Filter):
    """Helper to filter out some noise from lightning."""

    def filter(self, record):
        msg = record.getMessage()
        return not (
            "💡 Tip" in msg or "TPU available" in msg or "CUDA_VISIBLE_DEVICES" in msg
        )


logging.getLogger("lightning.pytorch.utilities.rank_zero").addFilter(TipFilter())
warnings.filterwarnings("ignore", ".*does not have many workers.*")
warnings.filterwarnings("ignore", r"(?s).*isinstance\(treespec, LeafSpec\)")

## Downloads
3 GiB in total, takes a while to download.

We use the [STL10 dataset](https://cs.stanford.edu/~acoates/stl10/). It is similar to CIFAR10 (10 classes: airplane, bird, car, cat, deer, dog, horse, monkey, ship, truck), but has higher resolution (96×96), and we only get 500 labeled images per class.<br>
Additionally, we get a much larger set of 100'000 unlabeled images, similar but sampled from a wider range of animals and vehicles.

In [ ]:
DATA_PATH = Path("data/")
CHECKPOINT_PATH = DATA_PATH / "SimCLR_saved_models/"

In [ ]:
print(len(torchvision.datasets.STL10(root=DATA_PATH, split="unlabeled", download=True)))
print(len(torchvision.datasets.STL10(root=DATA_PATH, split="train", download=True)))
print(len(torchvision.datasets.STL10(root=DATA_PATH, split="test", download=True)))

We also download some pretrained models for later (<100 MiB of downloads, ResNet18).

In [ ]:
%%bash
for s in \
    "SimCLR.ckpt" "ResNet.ckpt" "tensorboards/SimCLR/events.out.tfevents.SimCLR" "tensorboards/classification/ResNet/events.out.tfevents.ResNet" \
    "LogisticRegression_10.ckpt" "LogisticRegression_20.ckpt" "LogisticRegression_50.ckpt" "LogisticRegression_100.ckpt" "LogisticRegression_200.ckpt" "LogisticRegression_500.ckpt"
do
    target=data/SimCLR_saved_models/"$s"
    if [ ! -f "$target" ]; then
        wget -q https://raw.githubusercontent.com/phlippe/saved_models/main/tutorial17/"$s" -O "$target"
    fi
done

## Augmentations

Consider what specific augmentations we want to apply – this is the most crucial hyperparameter in SimCLR: it directly affects how the latent space is structured, and what patterns might be learned from the data. (figure credit - [Ting Chen and Geoffrey Hinton](https://ai.googleblog.com/2020/04/advancing-self-supervised-and-semi.html))

<center width="100%"><img src="https://github.com/phlippe/uvadlc_notebooks/blob/master/docs/tutorial_notebooks/tutorial17/simclr_data_augmentations.png?raw=1" width="800px" style="padding-top: 10px; padding-bottom: 10px"></center>

All of them can be used, but it turns out that two are more important: crop-and-resize, and color distortion. Interestingly, they only lead to strong performance when used together.
Random crop-and-resize can result in the following situations (among others):

<center width="100%"><img src="https://github.com/phlippe/uvadlc_notebooks/blob/master/docs/tutorial_notebooks/tutorial17/crop_views.svg?raw=1" width="400px" style="padding-top: 20px; padding-bottom: 0px"></center>

* Situation (a) requires the model to learn some sort of scale invariance.
* Situation (b) is more challenging and requires the model to learn about the whole structure of objects, not just local features.
* Still, without e.g. color distortion, just looking at overall color similarity would be sufficient to get good results (for the contrastive embedding task).

In [ ]:
augmentations = v2.Compose(
    [
        v2.ToImage(),
        v2.RandomHorizontalFlip(),
        v2.RandomResizedCrop(size=96),
        v2.RandomApply(
            [
                v2.ColorJitter(  # Reduced slightly from the original paper's settings.
                    brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1
                )
            ],
            p=0.8,
        ),
        v2.RandomGrayscale(p=0.2),
        v2.GaussianBlur(kernel_size=9),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize((0.5,), (0.5,)),
    ]
)

untransform = v2.Compose(
    [
        v2.Normalize(mean=[-1, -1, -1], std=[2, 2, 2]),
        v2.ToPILImage(),
    ]
)

We now need to prepare the data loading so that whenever an image is loaded from the dataset, we get two random augmentations of it.
We could do more than two (which would increase the number of _positive_ pairs), but two seems most efficient.


In [ ]:
class RepeatAndAugment:
    def __init__(self, augmentations: v2.Transform, n_views: int = 2) -> None:
        self.augmentations = augmentations
        self.n_views = n_views

    def __call__(self, x: Tensor) -> Tensor:
        """
        Input (C, H, W).
        Output (n_views, C, H, W) independently sampled augmentations of the image.
        """
        # TODO {
        # }
        return views

In [ ]:
unlabeled_dataset = torchvision.datasets.STL10(
    root=DATA_PATH,
    split="unlabeled",
    transform=RepeatAndAugment(augmentations, n_views=2),
)
labeled_dataset = torchvision.datasets.STL10(
    root=DATA_PATH,
    split="train",
    transform=RepeatAndAugment(augmentations, n_views=2),
)

In [ ]:
example_imgs, _lbl = next(
    iter(DataLoader(unlabeled_dataset, batch_size=5, num_workers=0))
)
example_imgs.shape

Finally, before starting with our implementation of SimCLR, let's look at some example image pairs sampled with our augmentations:

In [ ]:
fig, axs = plt.subplots(
    nrows=2, ncols=len(example_imgs), figsize=(len(example_imgs) * 2, 2 * 2)
)
fig.set_layout_engine("constrained", h_pad=0.04, w_pad=0.01, hspace=0.02, wspace=0.01)
for i, img_pair in enumerate(example_imgs):
    assert img_pair.shape == (2, 3, 96, 96)
    axs[0, i].imshow(untransform(img_pair[0]))
    axs[0, i].axis("off")
    axs[1, i].imshow(untransform(img_pair[1]))
    axs[1, i].axis("off")

## SimCLR implementation

At each iteration, we get for every image $x$ two differently augmented versions, which we refer to as $\tilde{x}_i$ and $\tilde{x}_j$. Both of these images are encoded into a one-dimensional feature vector, between which we want to maximize similarity which minimizes it to all other images in the batch. The encoder network is split into two parts:
* a base encoder network $f(\cdot)$ – a deep CNN (here: ResNet18) responsible for extracting a representation vector $h_i := f(\tilde{x}_i)$;
* a projection head $g(\cdot)$ – a small MLP  that maps the representation $h_i$ into a space where we apply the contrastive loss, $z_i := g(h_i)$.

<center width="100%"><img src="https://github.com/phlippe/uvadlc_notebooks/blob/master/docs/tutorial_notebooks/tutorial17/simclr_network_setup.svg?raw=1" width="350px"></center>

After finishing the training with contrastive learning, we forget the projection head $g(\cdot)$, and use $f(\cdot)$ as the pretrained feature extractor.
Using $g(\cdot)$ turns out to perform worse, likely because the representations $g(h_i)$ become invariant to many features like the color that can be important for downstream tasks.

We follow the original SimCLR paper by defining $g$ with a two-layer MLP with ReLU, though in the follow-up paper, [SimCLRv2](https://arxiv.org/abs/2006.10029), they mention that larger/wider MLPs can boost the performance considerably, so we increase the hidden dimension four times (on the other hand, deeper MLPs tended to overfit).

We use cosine similarity:
$$
\text{sim}(z_i,z_j) = \frac{z_i^\top \cdot z_j}{\|z_i\| \|z_j\|}
$$

To maximize similarity between positive pairs (i.e., two augmented versions of the same image) and minimize similarity to all other examples in the batch, we apply the InfoNCE loss, originally proposed by [Aaron van den Oord et al.](https://arxiv.org/abs/1807.03748) for contrastive learning.
Intuitively, to compute InfoNCE loss:
* apply logSoftMax to the vector of similarities between $x_i$ and all other images in the batch (positive or negative), and then
* returns minus the entries corresponding to positive pairs.

We assume `n_views=2`, so each image $x_i$ has a single positive paired image $x_{\sigma(i)}$.

$$
\ell_{i}=-\log \frac{\exp(\text{sim}(z_i,z_{\sigma(i)})/\tau)}{\sum_{k=1}^{2N}\mathbb{1}_{[k\neq i]}\exp(\text{sim}(z_i,z_k)/\tau)}=-\text{sim}(z_i,z_{\sigma(i)})/\tau+\log\left[\sum_{k=1}^{2N}\mathbb{1}_{[k\neq i]}\exp(\text{sim}(z_i,z_k)/\tau)\right]
$$

Note that $\sum_{k=1}^{2N}\mathbb{1}_{[k\neq i]}$ includes $\sigma(i)$, it only excludes $i$.
The hyperparameter $\tau$ is called temperature determining how peaked the softmax distribution is.
Since cosine similarity is bounded (as are many other similarity metrics), the temperature parameter allows us to balance the influence of many dissimilar image patches versus one similar patch.

In [ ]:
class SimCLR(L.LightningModule):
    def __init__(
        self,
        hidden_dim: int,
        lr: float,
        temperature: float,
        weight_decay: float,
        max_epochs: int = 500,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()  # Saved arguments to self.hparams.
        assert self.hparams.temperature > 0

        # Base model f(.) is a ResNet18 with 4 * hidden_dim output features.
        self.convnet = torchvision.models.resnet18(num_classes=4 * hidden_dim)

        # The MLP for g(.) consists of Linear->ReLU->Linear.
        self.convnet.fc = nn.Sequential(
            self.convnet.fc,  # Linear(ResNet backbone output, 4*hidden_dim)
            nn.ReLU(inplace=True),
            nn.Linear(4 * hidden_dim, hidden_dim),
        )

    def configure_optimizers(self):
        optimizer = optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
        )
        lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.hparams.max_epochs, eta_min=self.hparams.lr / 50
        )
        return [optimizer], [lr_scheduler]

    def _step(
        self, batch: tuple[Tensor, Tensor], mode: Literal["train", "val"] = "train"
    ) -> Tensor:
        imgs, _labels = batch
        B, n_views, C, H, W = imgs.shape

        # Encode all images.
        # TODO feats = ...  # shape (B * n_views, hidden_dim)

        # Normalize feature vectors so that dot product becomes cosine similarity.
        # TODO feats = ...

        # Compute pairwise cosine similarity matrix.
        # TODO cos_sim = ...  # shape (B * n_views, B * n_views)

        cos_sim = cos_sim / self.hparams.temperature

        # Mask out cosine similarity to itself.
        # TODO cos_sim[?] = ?

        loss_from_neg = torch.logsumexp(cos_sim, dim=-1).sum() / (B * n_views)

        # Extract cosine similarities for positive pairs (but not identity pairs).
        # TODO pos_sim = ..., you may assume n_views=2 for simplicity.

        loss_from_pos = -pos_sim.sum() / (B * n_views * (n_views - 1))

        nll = loss_from_pos + loss_from_neg

        # Logging loss
        self.log(f"{mode}_loss", nll)
        # with torch.no_grad():
        #     pos_sim = pos_sim.detach().clone()
        #     cos_sim = cos_sim.detach().clone()
        #     # Get ranking position of positive example
        #     pos_sim[:, torch.arange(n_views), torch.arange(n_views)] = -9e15
        #     cos_sim[torch.arange(B), :, torch.arange(B), :] = -9e15
        #     comb_sim = torch.cat(
        #         [
        #             # First n_views positive examples (with identity pairs masked out)
        #             pos_sim.view(B * n_views, n_views),
        #             # Then all examples (with positive pairs masked out).
        #             cos_sim.view(B * n_views, B * n_views),
        #         ],
        #         dim=-1,
        #     )
        #     sim_argsort = comb_sim.argsort(dim=-1, descending=True).argmin(dim=-1)

        #     # Logging ranking metrics
        #     self.log(f"{mode}_acc_top1", (sim_argsort < n_views).float().mean())
        #     self.log(f"{mode}_acc_top5", (sim_argsort < 5 * n_views).float().mean())
        #     self.log(f"{mode}_acc_mean_pos", 1 + sim_argsort.float().mean())

        return nll

    def training_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> Tensor:
        return self._step(batch, mode="train")

    def validation_step(self, batch: tuple[Tensor, Tensor], batch_idx: int) -> None:
        self._step(batch, mode="val")

In [ ]:
def sanity_check():
    model = SimCLR(hidden_dim=16, lr=1e-3, temperature=0.5, weight_decay=1e-4)
    trainer = L.Trainer(max_steps=2, devices=1, logger=False)
    trainer.fit(
        model,
        DataLoader(unlabeled_dataset, batch_size=3, num_workers=0),
        DataLoader(labeled_dataset, batch_size=3, num_workers=0),
    )


sanity_check()

## Training/loading pretrained model

Now that we have implemented SimCLR and the data loading pipeline, we are ready to train the model. We will use the same training function setup as usual. For saving the best model checkpoint, we track the metric `val_acc_top5`, which describes how often the correct image patch is within the top-5 most similar examples in the batch. This is usually less noisy than the top-1 metric, making it a better metric to choose the best model from.

Full SimCLR training is computationally expensive and is not part of this lab. Instead, we use a pretrained checkpoint. If `SimCLR.ckpt` is found, the notebook loads it automatically and skips training. The training code is kept only as a fallback and to show how SimCLR would normally be trained.

In [ ]:
def train_simclr(batch_size: int, max_epochs: int = 500, seed: int = 42, **kwargs):
    # Check whether pretrained model exists. If yes, load it and skip training
    pretrained_filename = CHECKPOINT_PATH / "SimCLR.ckpt"
    if pretrained_filename.is_file():
        print(f"Found pretrained model at {pretrained_filename}, loading...")
        return SimCLR.load_from_checkpoint(pretrained_filename)

    assert False, "No pretrained model found, training from scratch. Remove this assertion to proceed with training."
    L.seed_everything(seed)

    train_loader = DataLoader(
        unlabeled_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        pin_memory=True,
        num_workers=NUM_WORKERS,
    )
    val_loader = DataLoader(
        labeled_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        pin_memory=True,
        num_workers=NUM_WORKERS,
    )
    model = SimCLR(max_epochs=max_epochs, **kwargs)

    trainer = L.Trainer(
        default_root_dir=CHECKPOINT_PATH / "SimCLR",
        max_epochs=max_epochs,
        callbacks=[
            ModelCheckpoint(save_weights_only=True, mode="max", monitor="val_acc_top5"),
            LearningRateMonitor("epoch"),
        ],
    )
    # Artificial metric (for logging hyper-params) that we don't need.
    trainer.logger._default_hp_metric = None

    trainer.fit(model, train_loader, val_loader)
    # Load best checkpoint after training
    model = SimCLR.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)

    return model

A common observation in contrastive learning is that the larger the batch size, the better the models perform. A larger batch size allows us to compare each image to more negative examples, leading to overall smoother loss gradients. However, in our case, we experienced that a batch size of 256 was sufficient to get good results.

In [ ]:
simclr_model = train_simclr(
    batch_size=256,
    hidden_dim=128,
    lr=5e-4,
    temperature=0.07,
    weight_decay=1e-4,
    max_epochs=500,
)

To get an intuition of how training with contrastive learning behaves, we can take a look at the TensorBoard below:

In [ ]:
# %tensorboard --logdir ./data/SimCLR_saved_models/tensorboards/SimCLR/

<center width="100%"><img src="https://github.com/phlippe/uvadlc_notebooks/blob/master/docs/tutorial_notebooks/tutorial17/tensorboard_simclr.png?raw=1" width="1200px"></center>

One thing to note is that contrastive learning benefits a lot from long training. The shown plot above is from a training that took approx. 1 day on a NVIDIA TitanRTX. Training the model for even longer might reduce its loss further, but we did not experience any gains from it for the downstream task on image classification. In general, contrastive learning can also benefit from using larger models, if sufficient unlabeled data is available.

## Logistic Regression

After we have trained our model via contrastive learning, we can deploy it on downstream tasks and see how well it performs with little data. A common setup, which also verifies whether the model has learned generalized representations, is to perform Logistic Regression on the features. In other words, we learn a single, linear layer that maps the representations to a class prediction. Since the base network $f(\cdot)$ is not changed during the training process, the model can only perform well if the representations of $h$ describe all features that might be necessary for the task. Further, we do not have to worry too much about overfitting since we have very few parameters that are trained. Hence, we might expect that the model can perform well even with very little data.

First, let's implement a simple Logistic Regression setup for which we assume that the images already have been encoded in their feature vectors. If very little data is available, it might be beneficial to dynamically encode the images during training so that we can also apply data augmentations. However, the way we implement it here is much more efficient and can be trained within a few seconds. Further, using data augmentations did not show any significant gain in this simple setup.

In [ ]:
class LogisticRegression(L.LightningModule):
    def __init__(
        self,
        feature_dim: int,
        num_classes: int,
        lr: float,
        weight_decay: float,
        max_epochs: int = 100,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()
        self.model = nn.Linear(feature_dim, num_classes)

    def configure_optimizers(self):
        optimizer = optim.AdamW(
            self.model.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
        )
        lr_scheduler = optim.lr_scheduler.MultiStepLR(
            optimizer,
            milestones=[
                int(self.hparams.max_epochs * 0.6),
                int(self.hparams.max_epochs * 0.8),
            ],
            gamma=0.1,
        )
        return [optimizer], [lr_scheduler]

    def _step(self, batch, mode="train"):
        feats, labels = batch
        preds = self.model(feats)
        loss = F.cross_entropy(preds, labels)
        acc = (preds.argmax(dim=-1) == labels).float().mean()

        self.log(mode + "_loss", loss)
        self.log(mode + "_acc", acc)
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, mode="train")

    def validation_step(self, batch, batch_idx):
        self._step(batch, mode="val")

    def test_step(self, batch, batch_idx):
        self._step(batch, mode="test")

The data we use is the training and test set of STL10. The training contains 500 images per class, while the test set has 800 images per class.

In [ ]:
img_transforms = v2.Compose(
    [v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize((0.5,), (0.5,))]
)
train_img_data = torchvision.datasets.STL10(
    root=DATA_PATH, split="train", download=True, transform=img_transforms
)
test_img_data = torchvision.datasets.STL10(
    root=DATA_PATH, split="test", download=True, transform=img_transforms
)
print(len(train_img_data), len(test_img_data))

Next, we implement a small function to encode all images in our datasets. The output representations are then used as inputs to the Logistic Regression model.

In [ ]:
@torch.no_grad()
def prepare_data_features(model: nn.Module, dataset: Dataset) -> TensorDataset:
    """Run all images through the model (without .fc) to get their feature representations."""
    network = deepcopy(model.convnet)
    # Remove the projection head g(.) so that we use the encoder representation h.
    network.fc = nn.Identity()
    network.eval()
    network.to(DEVICE)

    data_loader = DataLoader(
        dataset, batch_size=64, num_workers=NUM_WORKERS, shuffle=False, drop_last=False
    )
    feats, labels = [], []
    for batch_imgs, batch_labels in tqdm(data_loader):
        batch_imgs = batch_imgs.to(DEVICE)
        batch_feats = network(batch_imgs)
        feats.append(batch_feats.detach().cpu())
        labels.append(batch_labels)

    feats = torch.cat(feats, dim=0)
    labels = torch.cat(labels, dim=0)

    # Sort images by labels
    labels, idxs = labels.sort()
    feats = feats[idxs]

    return TensorDataset(feats, labels)

Let's apply the function to both training and test set below.

In [ ]:
train_feats_simclr = prepare_data_features(simclr_model, train_img_data)
test_feats_simclr = prepare_data_features(simclr_model, test_img_data)

Finally, we can write a training function as usual. We evaluate the model on the test set every 10 epochs to allow early stopping, but the low frequency of the validation ensures that we do not overfit too much on the test set.

In [ ]:
def train_logreg(
    batch_size,
    train_feats_data,
    test_feats_data,
    model_suffix,
    max_epochs=100,
    **kwargs,
):
    trainer = L.Trainer(
        default_root_dir=CHECKPOINT_PATH / "LogisticRegression",
        max_epochs=max_epochs,
        callbacks=[
            ModelCheckpoint(save_weights_only=True, mode="max", monitor="val_acc"),
            LearningRateMonitor("epoch"),
        ],
        enable_progress_bar=False,
        check_val_every_n_epoch=10,
    )
    trainer.logger._default_hp_metric = None

    # Data loaders
    train_loader = DataLoader(
        train_feats_data,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        pin_memory=True,
        num_workers=0,
    )
    test_loader = DataLoader(
        test_feats_data,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        pin_memory=True,
        num_workers=0,
    )

    # Check whether pretrained model exists. If yes, load it and skip training
    pretrained_filename = CHECKPOINT_PATH / f"LogisticRegression_{model_suffix}.ckpt"
    if os.path.isfile(pretrained_filename):
        print(f"Found pretrained model at {pretrained_filename}, loading...")
        model = LogisticRegression.load_from_checkpoint(pretrained_filename)
    else:
        assert False, "No pretrained model found."
        L.seed_everything(42)
        model = LogisticRegression(**kwargs)
        trainer.fit(model, train_loader, test_loader)
        model = LogisticRegression.load_from_checkpoint(
            trainer.checkpoint_callback.best_model_path
        )

    # Test best model on train and validation set
    train_result = trainer.test(model, train_loader, verbose=False)
    test_result = trainer.test(model, test_loader, verbose=False)
    result = {"train": train_result[0]["test_acc"], "test": test_result[0]["test_acc"]}

    return model, result

Despite the training dataset of STL10 already only having 500 labeled images per class, we will perform experiments with even smaller datasets. Specifically, we train a Logistic Regression model for datasets with only 10, 20, 50, 100, 200, and all 500 examples per class. This gives us an intuition on how well the representations learned by contrastive learning can be transfered to a image recognition task like this classification. First, let's define a function to create the intended sub-datasets from the full training set:

In [ ]:
def get_smaller_dataset(
    original_dataset: TensorDataset, num_imgs_per_label: int, num_classes: int = 10
) -> TensorDataset:
    """
    Get a smaller dataset with only num_imgs_per_label images per class.

    Assumes that the original dataset is sorted by labels and that each class has at least num_imgs_per_label images.
    """
    return TensorDataset(
        *[
            t.unflatten(dim=0, sizes=(num_classes, -1))[:, :num_imgs_per_label].flatten(
                start_dim=0, end_dim=1
            )
            for t in original_dataset.tensors
        ]
    )

Next, let's run all models. Despite us training 6 models, this cell could be run within a minute or two without the pretrained models.

In [ ]:
results = {}
for num_imgs_per_label in [10, 20, 50, 100, 200, 500]:
    sub_train_set = get_smaller_dataset(train_feats_simclr, num_imgs_per_label)
    _, small_set_results = train_logreg(
        batch_size=64,
        train_feats_data=sub_train_set,
        test_feats_data=test_feats_simclr,
        model_suffix=num_imgs_per_label,
        feature_dim=train_feats_simclr.tensors[0].shape[1],
        num_classes=10,
        lr=1e-3,
        weight_decay=1e-3,
    )
    results[num_imgs_per_label] = small_set_results

Finally, let's plot the results.

In [ ]:
dataset_sizes = sorted([k for k in results])
test_scores = [results[k]["test"] for k in dataset_sizes]

fig = plt.figure(figsize=(6, 4))
plt.plot(dataset_sizes, test_scores, "--", marker=".")
plt.xscale("log")
plt.xticks(dataset_sizes, labels=dataset_sizes)
plt.title("STL10 classification over dataset size", fontsize=14)
plt.xlabel("Number of images per class")
plt.ylabel("Test accuracy")
plt.minorticks_off()
plt.show()

As expected, more data means higher accuracy. However, even with only 10 images per class, we can already classify more than 60% of the images correctly. With the full dataset, we achieve an accuracy of 81%.

For comparison, ResNet18 trained from scratch on 500 images achieves 73.31% on the test set (full training details [here](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial17/SimCLR.html#Baseline)).

## Questions

1. What is a positive/negative pair in SimCLR?
2. Why do we generate two independent, augmented views of the same image?
3. Why are random crop and color distortion useful together?
4. Why do we mask the diagonal of the similarity matrix?
5. What does the temperature parameter control?
6. Why do we remove the projection head before logistic regression?
7. Why can SimCLR help when only a few labels are available?